# Sheet 0.4: Matrix calculus and backprop

**Roadmap topic 0.4.** Read/watch first: Parr & Howard *The Matrix Calculus You Need for Deep Learning*, Karpathy's micrograd video and *Becoming a Backprop Ninja*, CS231n backprop notes.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Inside your engine and manual backward passes, **do not call** `.backward()` or `torch.autograd`. Plain tensor math only. CPU is fine.

In [ ]:
import math, torch
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

## Exercise 1: A tensor autograd engine

Build `class T` (a tensor with `.data`, `.grad`, and a backward closure), like micrograd but for whole tensors. Required ops:
`+`, `*` (both with **broadcasting**), `@`, `.sum(dim=None)`, `.mean()`, `.exp()`, `.log()`, `.relu()`, unary `-`, `-` (binary), and `.backward()` that does a topological sort and seeds the output gradient with ones.

The hard part is broadcasting: if `b` of shape `(1, n)` was broadcast to `(m, n)`, its gradient must be summed back to `(1, n)`. Write an `unbroadcast(grad, shape)` helper.

The check builds a 2-layer MLP with softmax cross-entropy using only your ops and compares every gradient with PyTorch.

In [ ]:
def unbroadcast(grad, shape):
    # YOUR CODE
    raise NotImplementedError

class T:
    def __init__(self, data, _children=()):
        self.data = data if isinstance(data, torch.Tensor) else torch.tensor(data)
        self.grad = torch.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
    # YOUR CODE: __add__, __radd__, __mul__, __rmul__, __neg__, __sub__, __matmul__,
    #            sum, mean, exp, log, relu, backward

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    B, D, H, K = 16, 8, 32, 5
    X = torch.randn(B, D); Y = torch.nn.functional.one_hot(torch.randint(0, K, (B,)), K).double()
    P = [torch.randn(D, H) * .3, torch.randn(1, H) * .1, torch.randn(H, K) * .3, torch.randn(K) * .1]
    mine = [T(p.clone()) for p in P]
    h = (T(X) @ mine[0] + mine[1]).relu()
    z = h @ mine[2] + mine[3]
    m = z.data.max(1, keepdim=True).values
    zs = z - T(m)
    lse = zs.exp().sum(1).log()
    loss = ((lse - (zs * T(Y)).sum(1)) ).mean()
    loss.backward()
    ref = [p.clone().requires_grad_() for p in P]
    hr = torch.relu(X @ ref[0] + ref[1]); zr = hr @ ref[2] + ref[3]
    lr = torch.nn.functional.cross_entropy(zr, Y.argmax(1)); lr.backward()
    assert abs(loss.data.item() - lr.item()) < 1e-10, "forward loss differs"
    for i, (a, r) in enumerate(zip(mine, ref)):
        assert a.grad.shape == r.grad.shape, f"param {i}: grad shape {tuple(a.grad.shape)} vs {tuple(r.grad.shape)}"
        assert torch.allclose(a.grad, r.grad, atol=1e-10), f"param {i}: gradient mismatch"
    x = T(torch.randn(3)); y = (x * x + x).sum(); y.backward()
    assert torch.allclose(x.grad, 2 * x.data + 1), "a node used twice must accumulate gradients"
    print("PASS")
_check1()

**Written (1a):** Why must `backward` visit nodes in reverse topological order instead of doing a simple recursive DFS from the output? Give a 3-node example that breaks the naive version.

**Your answer:**

## Exercise 2: Manual backward through LayerNorm and softmax cross-entropy

Forward pass for a batch:
```
mu, var = x.mean(-1), x.var(-1, unbiased=False)
xhat = (x - mu) / sqrt(var + eps)
h = xhat * gamma + beta
logits = h @ W
loss = cross_entropy(logits, targets)   # mean over batch
```
Derive and implement `manual_backward(x, gamma, beta, W, targets, eps)` returning `(dx, dgamma, dbeta, dW)` with **no autograd**. Use the compact forms: `dlogits = (softmax - onehot) / B` and the three-term LayerNorm formula. Write the derivation in the markdown cell below the code.

In [ ]:
def manual_backward(x, gamma, beta, W, targets, eps=1e-5):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    B, D, V = 32, 64, 100
    x = torch.randn(B, D) * 3 + 1; g = torch.randn(D); b = torch.randn(D); W = torch.randn(D, V) * .1
    t = torch.randint(0, V, (B,))
    ps = [p.clone().requires_grad_() for p in (x, g, b, W)]
    h = torch.nn.functional.layer_norm(ps[0], (D,), ps[1], ps[2], 1e-5)
    torch.nn.functional.cross_entropy(h @ ps[3], t).backward()
    mine = manual_backward(x, g, b, W, t)
    for name, a, r in zip(("dx", "dgamma", "dbeta", "dW"), mine, ps):
        assert a.shape == r.grad.shape, f"{name} shape"
        assert torch.allclose(a, r.grad, atol=1e-10), f"{name} mismatch (max err {(a - r.grad).abs().max():.2e})"
    print("PASS")
_check2()

**Your derivation of dx for LayerNorm:**

## Exercise 3: The softmax Jacobian, and why you never build it

1. `softmax_jacobian(z)`: the explicit `n×n` Jacobian of `softmax(z)` for a 1-D `z`.
2. `softmax_vjp(z, v)`: computes `Jᵀv` in **O(n)** time and memory, never forming `J`.

The check verifies both, then runs the VJP at `n = 50,257` (GPT-2's vocabulary), where the explicit Jacobian would need 20 GB in float64.

In [ ]:
def softmax_jacobian(z):
    # YOUR CODE
    raise NotImplementedError
def softmax_vjp(z, v):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    z = torch.randn(40); v = torch.randn(40)
    Jref = torch.autograd.functional.jacobian(lambda u: torch.softmax(u, 0), z)
    assert torch.allclose(softmax_jacobian(z), Jref, atol=1e-12), "Jacobian wrong"
    assert torch.allclose(softmax_vjp(z, v), Jref.T @ v, atol=1e-12), "VJP wrong"
    n = 50257; z = torch.randn(n) * 4; v = torch.randn(n)
    zz = z.clone().requires_grad_(); torch.softmax(zz, 0).backward(v)
    assert torch.allclose(softmax_vjp(z, v), zz.grad, atol=1e-12), "large-n VJP wrong"
    print("PASS")
_check3()

**Written (3a):** Reverse-mode autodiff costs about one extra forward pass per *output*, forward-mode about one per *input*. A loss is a scalar of millions of parameters. Which mode wins, and when would forward mode be the right choice?

**Your answer:**

## Exercise 4: A custom autograd Function for a fused SwiGLU-style op

`y = silu(x @ W1) * (x @ W2)` is the gate inside Llama's MLP. Autograd by default saves the intermediates `x@W1`, `silu(x@W1)` and `x@W2` for backward. For long sequences that memory adds up.

Write `class FusedSwiGLU(torch.autograd.Function)` whose `forward` saves **only** `x, W1, W2` (`ctx.save_for_backward`) and whose `backward` recomputes the intermediates and returns `(dx, dW1, dW2)` by hand. This trade of compute for memory is the idea behind activation checkpointing.

In [ ]:
class FusedSwiGLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W1, W2):
        # YOUR CODE
        raise NotImplementedError
    @staticmethod
    def backward(ctx, gy):
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    x = torch.randn(6, 5, requires_grad=True); W1 = torch.randn(5, 7, requires_grad=True); W2 = torch.randn(5, 7, requires_grad=True)
    assert torch.autograd.gradcheck(FusedSwiGLU.apply, (x, W1, W2)), "gradcheck failed"
    def saved_bytes(fn):
        tot = [0]
        def pack(t): tot[0] += t.numel() * t.element_size(); return t
        with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
            fn().sum()
        return tot[0]
    X = torch.randn(4096, 256, requires_grad=True); A = torch.randn(256, 1024, requires_grad=True); Bm = torch.randn(256, 1024, requires_grad=True)
    naive = saved_bytes(lambda: torch.nn.functional.silu(X @ A) * (X @ Bm))
    fused = saved_bytes(lambda: FusedSwiGLU.apply(X, A, Bm))
    assert fused < naive / 3, f"fused saves {fused/2**20:.0f} MiB vs naive {naive/2**20:.0f} MiB; save only the inputs"
    print(f"PASS  saved for backward: naive {naive/2**20:.0f} MiB, fused {fused/2**20:.0f} MiB")
_check4()

## Checkpoint

On paper, with shapes on every term, derive the backward pass of `L = CE(softmax(XW + b), y)` for a batch of `B` examples in under 20 minutes. Then check it against Exercise 2's `dlogits`. If you can't, redo *Becoming a Backprop Ninja* before 0.5.

---
### Hints for self-grading
- 1a: a node's gradient must be complete (all consumers have contributed) before it propagates further. Example: `a = 2x`, `b = 3a`, `c = a + b`. A naive DFS from `c` can visit `a` first and push its gradient (1 so far) into `x`, before the path through `b` has added its 3. `x.grad` ends up 2 instead of 8.
- 3a: reverse mode for scalar losses. Forward mode wins when there are few inputs and many outputs (e.g. a Jacobian-vector product in a sensitivity analysis or forward-mode JVPs for Hessian-vector products).